# 🧪 Expérimentation MLOps (Docker, MLflow, MinIO & Discord)
Ce notebook démontre comment s'intègre toute l'architecture.

In [2]:
import sys
import os
import boto3
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Ajout du chemin parent pour importer src
sys.path.append(os.path.abspath('../'))
from src.utils.discord_notify import notify_discord

# 1. Création automatique du Bucket S3 (MinIO) si inexistant
s3 = boto3.client('s3', 
                  endpoint_url='http://minio:9000', 
                  aws_access_key_id='root', 
                  aws_secret_access_key='root123456789')
try:
    s3.create_bucket(Bucket='mlflow')
    print("Bucket 'mlflow' créé dans MinIO.")
except Exception:
    print("Bucket 'mlflow' déjà existant.")

# 2. Connexion au Tracking Server MLflow (Conteneur Docker)
mlflow.set_tracking_uri('http://mlflow:5000')


Bucket 'mlflow' déjà existant.


In [3]:
notify_discord("Début d'une session Jupyter dans l'environnement Dockerisé...", "info")

try:
    # Chargement des données
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()
    
    # Entraînement
    
    with mlflow.start_run() as run:
        notify_discord(f"Entraînement du modèle (Run ID: `{run.info.run_id}`)...", "info")
        
        clf = RandomForestClassifier(n_estimators=100, max_depth=5)
        clf.fit(X_train, y_train)
        
        # Evaluation
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        
        notify_discord(f"Entraînement terminé avec succès !\n**Précision :** `{acc:.4f}`\n*Le modèle a été sauvegardé dans MinIO via MLflow.*", "success")

except Exception as e:
    notify_discord(f"Alerte lors de l'entraînement :\n```python\n{str(e)}\n```", "error")
    raise e

2026/05/11 19:52:04 INFO mlflow.tracking.fluent: Experiment with name 'Classification_Cancer' does not exist. Creating a new experiment.
2026/05/11 19:52:04 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.5.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/05/11 19:52:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run intelligent-crab-735 at: http://mlflow:5000/#/experiments/1/runs/26e1d8cb9b934d3cbaf67ff212ddab19
🧪 View experiment at: http://mlflow:5000/#/experiments/1
